# Splitmaa FunctionGemma Manual v4 Training

This notebook is a visual runner for the final `manual_v4` dataset workflow:

- inspect the frozen dataset report
- run strict validation and semantic audit
- convert/train FunctionGemma LoRA
- capture predictions from the trained adapter
- score the locked test set
- inspect failures and metrics

Run it from the repository root: `C:\\Users\\mario\\Documents\\projects\\Splitmaa`.

## 1. Setup

The notebook uses the existing `.venv-train` environment and repo scripts. If a cell fails because a package is missing, install it into `.venv-train`, not a random global Python.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "datasets" / "splitmaa_functiongemma" / "manual_v4" / "train.jsonl").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find Splitmaa repo root. Open this notebook from inside "
        "C:/Users/mario/Documents/projects/Splitmaa."
    )


REPO = find_repo_root(Path.cwd())
DATASET_DIR = REPO / "datasets" / "splitmaa_functiongemma" / "manual_v4"
TRAIN_JSONL = DATASET_DIR / "train.jsonl"
VALIDATION_JSONL = DATASET_DIR / "validation.jsonl"
TEST_JSONL = DATASET_DIR / "test.jsonl"
TRAIN_FG = DATASET_DIR / "train.functiongemma.jsonl"
VALIDATION_FG = DATASET_DIR / "validation.functiongemma.jsonl"
REPORTS_DIR = REPO / "reports" / "functiongemma_eval"
OUTPUT_DIR = REPO / "outputs" / "functiongemma-splitmaa-manual-v4-full-masked-len1600"

PY = REPO / ".venv-train" / "Scripts" / "python.exe"
if not PY.exists():
    PY = Path(sys.executable)

print("cwd:", Path.cwd())
print("repo:", REPO)
print("python:", PY)
print("dataset:", DATASET_DIR)
print("time:", datetime.now().isoformat(timespec="seconds"))


In [ ]:
def run(cmd, *, check=True, timeout=None):
    """Run a command from the repo root with UTF-8-safe, in-place progress output."""
    import codecs
    import html
    import json
    import re
    from IPython.display import HTML, display

    cmd = [str(part) for part in cmd]
    print("\n$", " ".join(cmd), flush=True)

    log_dir = REPO / "reports" / "functiongemma_eval" / "notebook_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"command_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    print("log:", log_path, flush=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUTF8"] = "1"
    env.setdefault("TERM", "xterm")

    process = subprocess.Popen(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
        env=env,
    )

    ansi_re = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
    progress_state = {}

    def clean(text):
        return ansi_re.sub("", text)

    def is_progress_line(text):
        stripped = text.strip()
        return bool(stripped) and ("|" in stripped and ("it/s" in stripped or "s/it" in stripped or "%" in stripped))

    def fmt_seconds(value):
        try:
            value = int(float(value))
        except (TypeError, ValueError):
            return "--"
        hours, remainder = divmod(value, 3600)
        minutes, seconds = divmod(remainder, 60)
        if hours:
            return f"{hours:d}h {minutes:02d}m"
        if minutes:
            return f"{minutes:d}m {seconds:02d}s"
        return f"{seconds:d}s"

    def metric(label, value, suffix=""):
        if value is None:
            value = "--"
        return (
            "<div style='min-width:120px;padding:10px 12px;border:1px solid #e5e7eb;"
            "border-radius:8px;background:#fff'>"
            f"<div style='font-size:11px;color:#6b7280;text-transform:uppercase'>{html.escape(label)}</div>"
            f"<div style='font-size:18px;font-weight:700;color:#111827'>{html.escape(str(value))}{suffix}</div>"
            "</div>"
        )

    def render_splitmaa_progress(payload):
        progress_state.update(payload)
        percent = max(0.0, min(100.0, float(progress_state.get("percent") or 0.0)))
        step = progress_state.get("step", 0)
        max_steps = progress_state.get("max_steps", 0)
        event = progress_state.get("event", "running")
        loss = progress_state.get("loss")
        eval_loss = progress_state.get("eval_loss")
        token_accuracy = progress_state.get("token_accuracy")
        if isinstance(token_accuracy, (int, float)):
            token_accuracy = f"{token_accuracy * 100:.2f}"
            token_suffix = "%"
        else:
            token_suffix = ""
        body = f"""
        <div style='font-family:Inter,Segoe UI,Arial,sans-serif;border:1px solid #d1d5db;border-radius:10px;padding:14px;background:#f9fafb;max-width:980px'>
          <div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:10px'>
            <div style='font-weight:800;color:#111827'>FunctionGemma LoRA training</div>
            <div style='font-size:13px;color:#4b5563'>{html.escape(str(event))} | step {html.escape(str(step))}/{html.escape(str(max_steps))} | epoch {html.escape(str(progress_state.get('epoch', '--')))}</div>
          </div>
          <div style='height:18px;background:#e5e7eb;border-radius:999px;overflow:hidden;border:1px solid #d1d5db'>
            <div style='height:100%;width:{percent:.2f}%;background:linear-gradient(90deg,#22c55e,#16a34a);transition:width .25s ease'></div>
          </div>
          <div style='display:flex;justify-content:space-between;font-size:12px;color:#4b5563;margin:6px 0 12px'>
            <span>{percent:.2f}% complete</span>
            <span>elapsed {fmt_seconds(progress_state.get('elapsed_seconds'))} | ETA {fmt_seconds(progress_state.get('eta_seconds'))}</span>
          </div>
          <div style='display:flex;gap:8px;flex-wrap:wrap'>
            {metric('train loss', loss)}
            {metric('eval loss', eval_loss)}
            {metric('token accuracy', token_accuracy, token_suffix)}
            {metric('grad norm', progress_state.get('grad_norm'))}
            {metric('lr', progress_state.get('learning_rate'))}
          </div>
        </div>
        """
        progress.update(HTML(body))

    progress = display(HTML("<pre style='margin:0;color:#6b7280'>waiting for process output...</pre>"), display_id=True)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    output_parts = []
    current_line = ""
    returncode = None

    def handle_line(line):
        line = clean(line)
        if not line.strip():
            return
        if line.startswith("SPLITMAA_PROGRESS "):
            try:
                render_splitmaa_progress(json.loads(line.split(" ", 1)[1]))
            except Exception as exc:
                print(f"Could not render progress line: {exc}", flush=True)
            return
        if is_progress_line(line):
            progress.update(HTML(f"<pre style='margin:0;color:#2563eb'>{html.escape(line)}</pre>"))
        else:
            print(line, flush=True)

    try:
        with log_path.open("ab") as log_file:
            log_file.write(("$ " + " ".join(cmd) + "\n").encode("utf-8", errors="replace"))
            while True:
                raw = process.stdout.read(256)
                if raw:
                    log_file.write(raw)
                    log_file.flush()
                    text = decoder.decode(raw)
                    output_parts.append(text)
                    for ch in text:
                        if ch == "\r":
                            handle_line(current_line)
                            current_line = ""
                        elif ch == "\n":
                            handle_line(current_line)
                            current_line = ""
                        else:
                            current_line += ch
                elif process.poll() is not None:
                    tail = decoder.decode(b"", final=True)
                    if tail:
                        output_parts.append(tail)
                        current_line += tail
                        log_file.write(tail.encode("utf-8", errors="replace"))
                    break

        if current_line.strip():
            handle_line(current_line)

        returncode = process.wait(timeout=timeout)
    finally:
        if process.poll() is None:
            process.kill()

    output = "".join(output_parts)
    if returncode == 0:
        progress_state["event"] = "finished"
        if progress_state:
            render_splitmaa_progress(progress_state)
        else:
            progress.update(HTML("<pre style='margin:0;color:#16a34a'>process finished successfully</pre>"))
    else:
        progress.update(HTML(f"<pre style='margin:0;color:#dc2626'>process failed with exit code {returncode}</pre>"))

    if check and returncode != 0:
        lines = clean(output).splitlines()
        print(f"\nLast log lines from {log_path}:", flush=True)
        print("\n".join(lines[-80:]), flush=True)
        raise RuntimeError(f"command failed with exit code {returncode}; see log: {log_path}")

    return subprocess.CompletedProcess(cmd, returncode, stdout=output, stderr=None)


## 2. Dataset Overview

In [ ]:
from collections import Counter

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

splits = {
    "train": load_jsonl(TRAIN_JSONL),
    "validation": load_jsonl(VALIDATION_JSONL),
    "test": load_jsonl(TEST_JSONL),
}

for split, rows in splits.items():
    counts = Counter(row["expected"]["arguments"]["workflowType"] for row in rows)
    print(split, len(rows), dict(counts))

print("total", sum(len(rows) for rows in splits.values()))


In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
except Exception as exc:
    print("Install optional notebook packages if you want charts:", exc)
else:
    rows = []
    for split, items in splits.items():
        for item in items:
            rows.append({
                "split": split,
                "workflowType": item["expected"]["arguments"]["workflowType"],
                "inputWords": len(item["input"].split()),
                "operations": len(item["expected"]["arguments"].get("operations", [])),
            })
    df = pd.DataFrame(rows)
    display(df.groupby(["split", "workflowType"]).size().unstack(fill_value=0))
    ax = df["workflowType"].value_counts().plot(kind="bar", title="Workflow Distribution", figsize=(10, 4))
    ax.set_xlabel("workflowType")
    ax.set_ylabel("rows")
    plt.tight_layout()
    plt.show()


In [ ]:
report_path = DATASET_DIR / "dataset_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "rows": report.get("rows"),
        "splitCounts": report.get("splitCounts"),
        "workflowCounts": report.get("workflowCounts"),
        "avgInputWords": report.get("avgInputWords"),
        "maxInputWords": report.get("maxInputWords"),
        "maxOperations": report.get("maxOperations"),
    }, indent=2))
else:
    print("No dataset report yet. Run the report cell below.")


## 3. Required Dataset Gates

These should pass before training. They are fast.

In [ ]:
run([
    PY,
    "tools/finetune/validate_splitmaa_dataset.py",
    "--strict-routing",
    TRAIN_JSONL,
    VALIDATION_JSONL,
    TEST_JSONL,
])


In [ ]:
run([
    PY,
    "tools/finetune/semantic_audit_dataset.py",
    TRAIN_JSONL,
    VALIDATION_JSONL,
    TEST_JSONL,
    "--report",
    DATASET_DIR / "semantic_audit_manual_v4.json",
    "--fail-on-blocking",
])


In [ ]:
run([
    PY,
    "tools/finetune/report_splitmaa_dataset.py",
    "--base",
    DATASET_DIR,
])


In [ ]:
run([
    PY,
    "tools/evals/run_eval.py",
    "--dataset",
    TEST_JSONL,
    "--self-test-with-expected",
    "--report",
    DATASET_DIR / "eval_self_test_report.json",
    "--failure-limit",
    "200",
])


## 4. Convert To FunctionGemma Format

Run this whenever the staging JSONL changes. The final files are what training uses.

In [ ]:
run([PY, "tools/finetune/convert_to_functiongemma.py", TRAIN_JSONL, TRAIN_FG])
run([PY, "tools/finetune/convert_to_functiongemma.py", VALIDATION_JSONL, VALIDATION_FG])


## 5. Train LoRA

Start with 3 epochs. If eval shows underfitting, try 5 epochs. Avoid jumping to 8 epochs until we inspect predictions.

In [ ]:
def latest_checkpoint(output_dir: Path):
    checkpoints = []
    if output_dir.exists():
        for child in output_dir.iterdir():
            if child.is_dir() and child.name.startswith("checkpoint-"):
                try:
                    checkpoints.append((int(child.name.split("-", 1)[1]), child))
                except ValueError:
                    pass
    return max(checkpoints)[1] if checkpoints else None


TRAIN_ARGS = [
    PY,
    "tools/finetune/train_functiongemma_sft.py",
    "--base-model", "google/functiongemma-270m-it",
    "--train", TRAIN_FG,
    "--validation", VALIDATION_FG,
    "--output-dir", OUTPUT_DIR,
    "--trainer-backend", "lean",
    "--training-mode", "full",
    "--epochs", "3",
    "--learning-rate", "2e-5",
    "--batch-size", "1",
    "--eval-batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--max-length", "1600",
    "--dtype", "bfloat16",
    "--gradient-checkpointing",
]

RESUME_CHECKPOINT = latest_checkpoint(OUTPUT_DIR)
if RESUME_CHECKPOINT:
    TRAIN_ARGS += ["--resume-from-checkpoint", RESUME_CHECKPOINT]
    print("resuming from:", RESUME_CHECKPOINT)
else:
    print("starting fresh")

print(" ".join(str(x) for x in TRAIN_ARGS))


In [ ]:
# This can take a while. Leave the notebook/kernel running.
run(TRAIN_ARGS)


## 6. Capture Predictions From Trained Adapter

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS = REPORTS_DIR / "manual_v4_lora_predictions.jsonl"

run([
    PY,
    "tools/evals/hf_peft_predictions.py",
    "--dataset",
    TEST_JSONL,
    "--adapter",
    OUTPUT_DIR,
    "--output",
    PREDICTIONS,
    "--device",
    "cuda",
    "--dtype",
    "bfloat16",
])


## 7. Score Locked Test Set

Initial acceptance thresholds:

- schema valid rate >= 0.95
- workflow accuracy >= 0.85
- operation sequence accuracy >= 0.80

In [ ]:
EVAL_REPORT = REPORTS_DIR / "manual_v4_lora_eval.json"

run([
    PY,
    "tools/evals/run_eval.py",
    "--dataset",
    TEST_JSONL,
    "--predictions",
    PREDICTIONS,
    "--report",
    EVAL_REPORT,
    "--fail-under-schema-valid",
    "0.95",
    "--fail-under-workflow",
    "0.85",
    "--fail-under-operation-sequence",
    "0.80",
    "--failure-limit",
    "100",
])


In [ ]:
if EVAL_REPORT.exists():
    eval_report = json.loads(EVAL_REPORT.read_text(encoding="utf-8"))
    keys = [
        "examples",
        "parseableRate",
        "schemaValidRate",
        "workflowAccuracy",
        "operationSequenceAccuracy",
        "exactIntentAccuracy",
        "leafArgumentAccuracy",
        "failures",
    ]
    print(json.dumps({key: eval_report.get(key) for key in keys}, indent=2))
else:
    print("No eval report yet.")


## 8. Inspect Failures

If the eval fails thresholds, inspect examples instead of changing hyperparameters blindly.

In [ ]:
if EVAL_REPORT.exists():
    eval_report = json.loads(EVAL_REPORT.read_text(encoding="utf-8"))
    failures = eval_report.get("failureExamples") or eval_report.get("failuresDetail") or []
    print("failure rows:", len(failures))
    for item in failures[:10]:
        print("\n---")
        print(json.dumps(item, indent=2)[:3000])
else:
    print("Run scoring first.")


## 9. Decision Guide

- If schema validity is low: inspect raw predictions and parser behavior first.
- If workflow accuracy is low: review routing failures and add/repair rows only if the label is wrong or under-covered.
- If operation sequence is low: check multi-step rows and long commands.
- If everything passes: export/merge the adapter and move toward a mobile-loadable FunctionGemma artifact.